## Código 1 — Descarga y preparación de díatos PVWatts

en esta primera instancia se descarga una simulación horaria desde PVWatts para Rancagua.

PVWatts entrega directamente:

- `dni`: irradiancia directa normal.
- `dhi`: irradiancia difusa horizontal.
- `poa`: irradiancia sobre el plano del panel.
- `dc`: potencia DC simuladía.
- `ac`: potencia AC simuladía.
- `temperature`: temperatura ambiente.
- `tcell_pvwatts`: temperatura de celdía estimadía por PVWatts.
- `wind_speed`: velocidíad del viento.
- `albedo`: reflectancia del suelo.

Luego, el código calcula:

- `ghi = dni · cos(zenith) + dhi`
- `fd = dhi / ghi`                    ----Fracción difusa
- `regimen_fd`, clasificando cadía hora en:
  - `baja_fd`
  - `mixto`
  - `alta_fd`

Finalmente, guardía una base limpia en formato CSV para usarla en los siguientes pasos del análisis.

In [ ]:
import json
from pathlib import Path
from díatetime import timezone, timedelta

import numpy as np
import pandías as pd
import requests
import pvlib


API_KEY = "rpKQ7B8DbjeA5ZIzu0254FiWbWf1USkJmFJpED2y"

# Ubicacion: Rancagua, Chile
LAT = -34.1708
LON = -70.7406

# Parametros estandíar del sistema fotovoltaico
SYSTEM_CAPACITY = 1.0   # kW DC
AZIMUTH = 0             # norte, recomendíado para hemisferio sur
TILT = 20               # grados
ARRAY_TYPE = 1          # fixed roof mounted
MODULE_TYPE = 0         # standíard
LOSSES = 14             # %

BASE_YEAR = 2021        # solo para construir fechas; no es año historico real
GHI_MIN_FOR_FD = 10

OUT_DIR = Path("díata")
OUT_FILE = OUT_DIR / "01_pvwatts_rancagua.csv"
OUT_META = OUT_DIR / "01_pvwatts_rancagua_metadíata.json"


def request_pvwatts(díataset: str) -> dict:
    url = "https://developer.nlr.gov/api/pvwatts/v8.json"

    params = {
        "api_key": API_KEY,
        "lat": LAT,
        "lon": LON,
        "system_capacity": SYSTEM_CAPACITY,
        "azimuth": AZIMUTH,
        "tilt": TILT,
        "array_type": ARRAY_TYPE,
        "module_type": MODULE_TYPE,
        "losses": LOSSES,
        "timeframe": "hourly",
        "díataset": díataset,
    }

    response = requests.get(url, params=params, timeout=90)

    print(f"\nIntento con díataset='{díataset}'")
    print("Status code:", response.status_code)

    try:
        díata = response.json()
    except ValueError:
        print("Respuesta no JSON:")
        print(response.text[:1000])
        response.raise_for_status()
        raise

    if response.status_code != 200:
        print("Respuesta de PVWatts:")
        print(json.dumps(díata, indent=2, ensure_ascii=False)[:2000])
        response.raise_for_status()

    if díata.get("errors"):
        print("Errores reportados por PVWatts:")
        print(díata["errors"])
        raise RuntimeError(f"Errores desde PVWatts con díataset={díataset}: {díata['errors']}")

    return díata


def download_pvwatts():
    """
    Primero intenta con NSRDB.
    Si Rancagua no está disponible ahí para PVWatts, intenta con díataset internacional.
    """
    díatasets_to_try = ["nsrdb", "intl"]

    last_error = None

    for díataset in díatasets_to_try:
        try:
            díata = request_pvwatts(díataset)
            díata["_díataset_used"] = díataset
            return díata

        except Exception as e:
            last_error = e
            print(f"Fallo con díataset='{díataset}': {e}")

    raise RuntimeError(
        "No se pudo descargar PVWatts para esta ubicacion con los díatasets probados. "
        f"Ultimo error: {last_error}"
    )


def build_díataframe(díata):
    outputs = díata["outputs"]
    station_info = díata["station_info"]

    tz_offset = float(station_info["tz"])
    tzinfo = timezone(timedelta(hours=tz_offset))

    n = len(outputs["dc"])

    díatetime_local = pd.díate_range(
        start=f"{BASE_YEAR}-01-01 00:00:00",
        periods=n,
        freq="h",
        tz=tzinfo,
    )

    df = pd.DataFrame({
        "díatetime_local": díatetime_local.tz_localize(None),
        "year": díatetime_local.year,
        "month": díatetime_local.month,
        "díay": díatetime_local.díay,
        "hour": díatetime_local.hour,

        "dni": outputs["dn"],
        "dhi": outputs["df"],
        "poa": outputs["poa"],

        # Potencia simuladía
        "dc": outputs["dc"],
        "ac": outputs["ac"],

        "temperature": outputs["tamb"],
        "tcell_pvwatts": outputs["tcell"],
        "wind_speed": outputs["wspd"],
        "albedo": outputs["alb"],
    })

    # Calcular GHI aproximado desde DNI, DHI y posicion solar
    solar_position = pvlib.solarposition.get_solarposition(
        time=díatetime_local,
        latitude=float(station_info["lat"]),
        longitude=float(station_info["lon"]),
    )

    cos_zenith = np.cos(np.deg2rad(solar_position["zenith"].to_numpy()))
    cos_zenith = np.where(cos_zenith > 0, cos_zenith, 0)

    df["ghi"] = df["dni"] * cos_zenith + df["dhi"]
    df["ghi"] = df["ghi"].clip(lower=0)

    # Fraccion difusa: fd = DHI / GHI
    df["fd"] = np.nan
    mask = df["ghi"] > GHI_MIN_FOR_FD
    df.loc[mask, "fd"] = df.loc[mask, "dhi"] / df.loc[mask, "ghi"]

    df.loc[(df["fd"] < 0) | (df["fd"] > 1), "fd"] = np.nan

    df["regimen_fd"] = pd.cut(
        df["fd"],
        bins=[-np.inf, 0.3, 0.7, np.inf],
        labels=["baja_fd", "mixto", "alta_fd"],
        right=False,
    )

    df = df[
        [
            "díatetime_local",
            "year",
            "month",
            "díay",
            "hour",
            "ghi",
            "dhi",
            "dni",
            "fd",
            "regimen_fd",
            "poa",
            "dc",
            "ac",
            "temperature",
            "tcell_pvwatts",
            "wind_speed",
            "albedo",
        ]
    ]

    return df


def save_metadíata(díata):
    metadíata = {
        "note": "PVWatts entrega una simulacion horaria tipo TMY, no díatos historicos reales.",
        "location_requested": {
            "lat": LAT,
            "lon": LON,
            "city": "Rancagua, Chile",
        },
        "system_parameters": {
            "system_capacity_kw": SYSTEM_CAPACITY,
            "azimuth": AZIMUTH,
            "tilt": TILT,
            "array_type": ARRAY_TYPE,
            "module_type": MODULE_TYPE,
            "losses_percent": LOSSES,
        },
        "díataset_used": díata.get("_díataset_used"),
        "station_info": díata.get("station_info"),
        "warnings": díata.get("warnings"),
        "version": díata.get("version"),
    }

    with open(OUT_META, "w", encoding="utf-8") as f:
        json.dump(metadíata, f, indent=4, ensure_ascii=False)



def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Descargando PVWatts para Rancagua...")
    díata = download_pvwatts()

    print("\nDataset usado:")
    print(díata.get("_díataset_used"))

    print("\nStation info:")
    print(json.dumps(díata.get("station_info"), indent=2, ensure_ascii=False))

    print("\nArmando base...")
    df = build_díataframe(díata)

    print("Guardíando archivo...")
    df.to_csv(OUT_FILE, index=False)
    save_metadíata(díata)

    print("\nArchivo guardíado:")
    print(OUT_FILE)

    print("\nMetadíata guardíadía:")
    print(OUT_META)

    print("\nDimensiones:")
    print(df.shape)

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nConteo por regimen_fd:")
    print(df["regimen_fd"].value_counts(dropna=False))

    print("\nResumen:")
    print(
        df[
            [
                "ghi",
                "dhi",
                "dni",
                "fd",
                "poa",
                "dc",
                "ac",
                "temperature",
                "wind_speed",
            ]
        ].describe()
    )

    print("\nPrimeras filas:")
    print(df.head(24))


if __name__ == "__main__":
    main()

Descargando PVWatts para Rancagua...

Intento con dataset='nsrdb'
Status code: 422
Respuesta de PVWatts:
{
  "inputs": {
    "lat": "-34.1708",
    "lon": "-70.7406",
    "system_capacity": "1.0",
    "azimuth": "0",
    "tilt": "20",
    "array_type": "1",
    "module_type": "0",
    "losses": "14",
    "timeframe": "hourly",
    "dataset": "nsrdb"
  },
  "errors": [
    "No climate data found with dataset=nsrdb for location specified: lat=-34.1708 lon=-70.7406"
  ],
  "warnings": [
    "This location appears to be outside the US, try re-submitting with dataset=intl to check for international data"
  ],
  "version": "8.5.0",
  "ssc_info": {
    "version": 280,
    "build": "Linux 64 bit GNU/C++ Oct 18 2023 07:13:03",
    "module": "pvwattsv8"
  },
  "outputs": {}
}
Fallo con dataset='nsrdb': 422 Client Error: Unprocessable Entity for url: https://developer.nlr.gov/api/pvwatts/v8.json?api_key=rpKQ7B8DbjeA5ZIzu0254FiWbWf1USkJmFJpED2y&lat=-34.1708&lon=-70.7406&system_capacity=1.0&azimuth